<a href="https://colab.research.google.com/github/unclesam243/reinforcement_learning_online_msds/blob/main/03_solving_mdps/lab_value_iteration_grid_world.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Lab: Value Iteration in a Grid World

### University of Virginia
### Reinforcement Learning

### Enock Samalenge
#### Last updated: June 9, 2026

---

#### Instructions:

Implement value iteration for a $4 \times 3$ gridworld environment. This will measure the value of each state. A robot in this world can make discrete moves: one step up, down, left or right. These actions are deterministic, meaning that the action selected will be taken with probability 1. There is a terminal state with reward +1 in the bottom right corner. All other states have reward 0. The discount factor is 0.9. Use tolerance $\theta=0.01$. Show all code and results.

**Note**: Do not use libraries from `networkx`, `gym`, `gymnasium` when solving this problem.

#### Total Points: 12

---

#### 1) **(POINTS: 2)** As part of your solution, create a GridWorld class with these attributes:

- `nrows` : number of rows in the grid
- `ncols` : number of columns in the grid

and these methods:

- `value_iteration()` with behavior described in [2] below
- `get_reward()` : given the agent row and column, return the reward

The class may include additional attributes and methods as well.

Create an instance using the class, and call `nrows`, `ncols`, and `get_reward()` to verify correctness.

You will not be graded on the implementation of `value_iteration()` for this problem.

#### 2) **(POINTS: 8)** Here, you will be graded on the implementation of `value_iteration()`.
Call `value_iteration()` to calculate and return the value function array. For each sweep over the states, have the function print out the intermediate array.


#### Enter all code here (you may also use multiple cells)

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd



#### 1) Create and test the class

In [2]:
class GridWorld:
  def __init__(self, nrows, ncols):
    self.nrows = nrows
    self.ncols = ncols

    # starting position
    self.start_state = np.array([0,0]) # starting at 0,0

    #Actions definition:
    self.actions = np.array([[-1, 0], #up
                             [1, 0], #down
                            [0, -1], #left
                             [0, 1] #right
                             ])

    self.action_names = ["up", "down", "left", "right"]

    # state definition:
    self.states = np.array([[i,j] for i in range(nrows) for j in range(ncols)])
    self.terminal_state = np.array([nrows-1, ncols-1])
    self.agent_pos = list(self.start_state) # current position
    self.reset()

  def reset(self):
      """Resets the environment and returns the initial state."""
      self.agent_pos = list(self.start_state)
      return self.agent_pos

  def get_reward(self, state):
    """Returns the reward for a given state."""
    if np.array_equal(state, self.terminal_state):
      return 1
    else:
      return 0

  def get_next_state(self, current_state, action):
    """Calculates the next state given a current state and an action, handling boundary conditions."""
    next_row = current_state[0] + action[0]
    next_col = current_state[1] + action[1]

    # Boundary conditions: if agent tries to move off grid, it stays in current state
    if not (0 <= next_row < self.nrows and 0 <= next_col < self.ncols):
        return current_state # Stays in the current state
    else:
        return np.array([next_row, next_col])


  def value_iteration(self, theta=0.01, discount_factor=0.9):
      """Performs value iteration and returns the optimal value function."""
      V = np.zeros((self.nrows, self.ncols)) # Initialize value function V(s) to 0 for all states

      print("Initial Value Function V:")
      print(V)
      print("-" * 30)

      iteration = 0
      while True:
          delta = 0 # Initialize delta
          V_old = V.copy() # Store old V for delta calculation to track updates

          for s_row, s_col in self.states: # Iterate through all states (row, col)
              s = np.array([s_row, s_col])
              v_old_s = V[s_row, s_col] # Value of state s before update

              # Calculate the value for each possible action from state s
              action_values = []
              for action in self.actions:
                  next_s = self.get_next_state(s, action)
                  reward = self.get_reward(next_s) # Reward based on the state reached
                  # Value of the next state (next_s) from the previous iteration (V_old)
                  value_of_next_state = V_old[next_s[0], next_s[1]]
                  action_value = reward + discount_factor * value_of_next_state
                  action_values.append(action_value)

              # Update V(s) with the maximum action value (Bellman optimality equation)
              if action_values: # Ensure there are possible actions
                V[s_row, s_col] = np.max(action_values)
              else:
                V[s_row, s_col] = v_old_s # No actions, value remains same

              # Update delta
              delta = max(delta, np.abs(v_old_s - V[s_row, s_col]))

          iteration += 1
          print(f"Value Function V after iteration {iteration}:")
          print(V)
          print("-" * 30)

          # Check for convergence
          if delta < theta:
              break

      print(f"Value Iteration converged after {iteration} iterations. Final delta: {delta:.4f}")
      return V



#### 2) Run value iteration

In [4]:
gw = GridWorld(nrows=4, ncols=3)
optimal_V = gw.value_iteration()

print("\nOptimal Value Function (V):")
print(optimal_V)

Initial Value Function V:
[[0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]]
------------------------------
Value Function V after iteration 1:
[[0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 1.]
 [0. 1. 1.]]
------------------------------
Value Function V after iteration 2:
[[0.  0.  0. ]
 [0.  0.  0.9]
 [0.  0.9 1.9]
 [0.9 1.9 1.9]]
------------------------------
Value Function V after iteration 3:
[[0.   0.   0.81]
 [0.   0.81 1.71]
 [0.81 1.71 2.71]
 [1.71 2.71 2.71]]
------------------------------
Value Function V after iteration 4:
[[0.    0.729 1.539]
 [0.729 1.539 2.439]
 [1.539 2.439 3.439]
 [2.439 3.439 3.439]]
------------------------------
Value Function V after iteration 5:
[[0.6561 1.3851 2.1951]
 [1.3851 2.1951 3.0951]
 [2.1951 3.0951 4.0951]
 [3.0951 4.0951 4.0951]]
------------------------------
Value Function V after iteration 6:
[[1.24659 1.97559 2.78559]
 [1.97559 2.78559 3.68559]
 [2.78559 3.68559 4.68559]
 [3.68559 4.68559 4.68559]]
------------------------------
Value Function V a

#### 3) **(POINTS: 2)** Based on the value function: After the agent has moved right or down, does it ever make sense for it to backtrack (move up or left)? Explain your reasoning.

Based on the value function, does it ever make sense for the agent to backtrack (move up or left) after it has moved right or down? Explain your reasoning.

No, backtracking (moving up or left) after going right or down generally doesn't make sense. The `optimal_V` array clearly shows that values increase as you move towards the bottom-right terminal state. Moving away from the goal leads to states with lower optimal values, which is suboptimal if the agent's goal is to maximize reward. Essentially, to get the most points, you always want to follow the path of increasing values.